In [ ]:
# 라이브러리 실행 확인용
import pandas as pd

In [ ]:
# 챔피언별 팀포지션별 픽률
import pandas as pd

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 챔피언 이름과 팀 포지션 데이터 결측치 제거
df_valid = df.dropna(subset=['championName', 'teamPosition'])

# 3. 행(챔피언) x 열(포지션) 피벗 테이블 생성
# normalize='index'를 사용하여 각 챔피언(행)의 포지션 비율 총합이 100%가 되도록 계산합니다.
pivot_ratio = pd.crosstab(
    index=df_valid['championName'], 
    columns=df_valid['teamPosition'], 
    normalize='index'
) * 100

# 4. 소수점 둘째 자리까지 반올림 및 빈 값은 0으로 채우기
pivot_ratio = pivot_ratio.round(2).fillna(0)

# 5. 열(팀 포지션) 순서를 원하는 대로 지정 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# reindex를 통해 설정한 순서대로 '열(columns)'을 재배열합니다.
pivot_ratio = pivot_ratio.reindex(columns=desired_order).fillna(0)

# 6. 결과 확인 (상위 15개 챔피언 출력)
print("챔피언(행) x 팀 포지션(열) 픽률 데이터:")
display(pivot_ratio.head(15))

# 7. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_ratio_kr.csv"
pivot_ratio.to_csv(output_path, encoding='utf-8-sig')

print(f"\n행과 열이 바뀐 데이터가 성공적으로 저장되었습니다:\n{output_path}")

챔피언(행) x 팀 포지션(열) 픽률 데이터:


teamPosition,TOP,JUNGLE,MIDDLE,BOTTOM,UTILITY
championName,,,,,
Aatrox,81.88,14.24,3.13,0.12,0.64
Ahri,1.16,0.06,96.45,0.12,2.21
Akali,17.18,0.00,82.47,0.18,0.18
Akshan,4.65,0.00,89.92,3.10,2.33
Alistar,2.36,0.34,1.24,0.34,95.73
Ambessa,79.53,14.73,5.24,0.33,0.17
Amumu,0.68,79.11,0.00,0.00,20.21
Anivia,7.66,0.23,68.24,1.35,22.52
Annie,6.47,0.00,79.12,1.76,12.65



행과 열이 바뀐 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_ratio_kr.csv


In [ ]:
# 챔피언별 팀포지션별 픽률 승률
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거 및 데이터 타입 변환
# 승률 계산을 위해 'win' 컬럼이 필요합니다.
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win']).copy()

# win 컬럼이 True/False 형태일 수 있으므로 1과 0의 숫자로 변환합니다.
df_valid['win'] = df_valid['win'].astype(int)

# 3. 픽률 계산 (각 챔피언 내 포지션 비중)
pick_rate = pd.crosstab(
    index=df_valid['championName'], 
    columns=df_valid['teamPosition'], 
    normalize='index'
) * 100

# 4. 승률 계산 (해당 챔피언이 해당 포지션으로 갔을 때의 승리 비율)
win_rate = pd.pivot_table(
    data=df_valid,
    values='win',
    index='championName',
    columns='teamPosition',
    aggfunc='mean' # 평균을 구하면 승률이 됩니다. (예: 0.52 -> 52%)
) * 100

# 5. 원하는 열 순서 지정
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 열 순서 맞추기 및 결측치 0으로 채우기
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

# 6. 픽률과 승률을 하나의 데이터프레임으로 병합
# 새로운 빈 데이터프레임을 만들고 지정된 순서대로 픽률과 승률 열을 번갈아 추가합니다.
result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)

# 7. 결과 확인 (상위 15개 챔피언)
print("챔피언(행) x 팀 포지션별 픽률 및 승률 데이터:")
display(result_df.head(15))

# 8. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_pick_win_rate_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n포지션별 픽률과 승률이 포함된 데이터가 저장되었습니다:\n{output_path}")

챔피언(행) x 팀 포지션별 픽률 및 승률 데이터:


,TOP_픽률(%),TOP_승률(%),JUNGLE_픽률(%),JUNGLE_승률(%),MIDDLE_픽률(%),MIDDLE_승률(%),BOTTOM_픽률(%),BOTTOM_승률(%),UTILITY_픽률(%),UTILITY_승률(%)
championName,,,,,,,,,,
Aatrox,81.88,48.66,14.24,48.37,3.13,50.00,0.12,100.00,0.64,27.27
Ahri,1.16,60.00,0.06,0.00,96.45,48.55,0.12,50.00,2.21,23.68
Akali,17.18,51.92,0.00,0.00,82.47,48.48,0.18,0.00,0.18,25.00
Akshan,4.65,50.00,0.00,0.00,89.92,48.28,3.10,12.50,2.33,50.00
Alistar,2.36,28.57,0.34,33.33,1.24,36.36,0.34,0.00,95.73,50.82
Ambessa,79.53,47.70,14.73,51.98,5.24,47.62,0.33,0.00,0.17,100.00
Amumu,0.68,25.00,79.11,55.19,0.00,0.00,0.00,0.00,20.21,55.93
Anivia,7.66,44.12,0.23,0.00,68.24,53.80,1.35,50.00,22.52,47.00
Annie,6.47,40.91,0.00,0.00,79.12,49.07,1.76,50.00,12.65,53.49



포지션별 픽률과 승률이 포함된 데이터가 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_pick_win_rate_kr.csv


In [ ]:
# 챔피언별 팀포지션별 매치수 픽률 승률
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거 및 데이터 타입 변환
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win']).copy()
df_valid['win'] = df_valid['win'].astype(int)

# 3. 지표 계산
# 매치 수 계산 (단순 발생 횟수)
match_count = pd.crosstab(
    index=df_valid['championName'], 
    columns=df_valid['teamPosition']
)

# 픽률 계산 (각 챔피언 내 포지션 비중)
pick_rate = pd.crosstab(
    index=df_valid['championName'], 
    columns=df_valid['teamPosition'], 
    normalize='index'
) * 100

# 승률 계산 (해당 챔피언이 해당 포지션으로 갔을 때의 승리 비율)
win_rate = pd.pivot_table(
    data=df_valid,
    values='win',
    index='championName',
    columns='teamPosition',
    aggfunc='mean'
) * 100

# 4. 원하는 열 순서 지정
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 열 순서 맞추기 및 결측치 0으로 채우기
match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

# 5. 매치 수, 픽률, 승률을 하나의 데이터프레임으로 병합
result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    # 매치 수는 정수형으로 변환
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)

# 6. 결과 확인 (상위 10개 챔피언)
print("챔피언(행) x 팀 포지션별 매치수, 픽률, 승률 데이터:")
display(result_df.head(10))

# 7. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_teamposition_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n매치수, 픽률, 승률이 포함된 통합 데이터가 저장되었습니다:\n{output_path}")

챔피언(행) x 팀 포지션별 매치수, 픽률, 승률 데이터:


,TOP_매치수,TOP_픽률(%),TOP_승률(%),JUNGLE_매치수,JUNGLE_픽률(%),JUNGLE_승률(%),MIDDLE_매치수,MIDDLE_픽률(%),MIDDLE_승률(%),BOTTOM_매치수,BOTTOM_픽률(%),BOTTOM_승률(%),UTILITY_매치수,UTILITY_픽률(%),UTILITY_승률(%)
championName,,,,,,,,,,,,,,,
Aatrox,1414,81.88,48.66,246,14.24,48.37,54,3.13,50.00,2,0.12,100.00,11,0.64,27.27
Ahri,20,1.16,60.00,1,0.06,0.00,1656,96.45,48.55,2,0.12,50.00,38,2.21,23.68
Akali,391,17.18,51.92,0,0.00,0.00,1877,82.47,48.48,4,0.18,0.00,4,0.18,25.00
Akshan,12,4.65,50.00,0,0.00,0.00,232,89.92,48.28,8,3.10,12.50,6,2.33,50.00
Alistar,21,2.36,28.57,3,0.34,33.33,11,1.24,36.36,3,0.34,0.00,852,95.73,50.82
Ambessa,956,79.53,47.70,177,14.73,51.98,63,5.24,47.62,4,0.33,0.00,2,0.17,100.00
Amumu,4,0.68,25.00,462,79.11,55.19,0,0.00,0.00,0,0.00,0.00,118,20.21,55.93
Anivia,34,7.66,44.12,1,0.23,0.00,303,68.24,53.80,6,1.35,50.00,100,22.52,47.00
Annie,22,6.47,40.91,0,0.00,0.00,269,79.12,49.07,6,1.76,50.00,43,12.65,53.49



매치수, 픽률, 승률이 포함된 통합 데이터가 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_teamposition_kr.csv


In [ ]:
# 팀포지션별 평균 골드,딜량,시야점수
import pandas as pd

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 포지션 데이터 결측치 제거
df_valid = df.dropna(subset=['teamPosition']).copy()

# 3. 분석할 컬럼 선택 (골드, 챔피언 대상 피해량, 시야 점수)
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']

# 4. 포지션별 평균값 계산
position_stats = df_valid.groupby('teamPosition')[metrics].mean()

# 5. 보기 좋게 데이터 정제
# 소수점 첫째 자리까지 반올림
position_stats = position_stats.round(1)

# 컬럼명 직관적으로 변경
position_stats.rename(columns={
    'goldEarned': '평균 획득 골드',
    'totalDamageDealtToChampions': '평균 챔피언 피해량(딜량)',
    'visionScore': '평균 시야 점수'
}, inplace=True)

# 6. 포지션 순서 정렬 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
position_stats = position_stats.reindex(desired_order)

# 인덱스 이름(teamPosition)도 한글로 변경해 주면 더 깔끔합니다.
position_stats.index.name = '포지션'

# 7. 결과 확인
print("포지션별 평균 골드/딜/시야 분포:")
display(position_stats)

# 8. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\position_average_stats_kr.csv"
position_stats.to_csv(output_path, encoding='utf-8-sig')

print(f"\n포지션별 평균 분포 데이터가 성공적으로 저장되었습니다:\n{output_path}")

포지션별 평균 골드/딜/시야 분포:


,평균 획득 골드,평균 챔피언 피해량(딜량),평균 시야 점수
포지션,,,
TOP,11356.1,23112.3,17.7
JUNGLE,11846.5,19178.3,26.1
MIDDLE,11384.2,23183.0,19.5
BOTTOM,12480.5,23870.8,17.5
UTILITY,8490.8,13116.5,62.6



포지션별 평균 분포 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\position_average_stats_kr.csv


In [ ]:
# 팀포지션별 승패별 평균 골드,딜량,시야,팀내 비중
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 결측치 제거 및 타입 변환
df_valid = df.dropna(subset=['teamPosition', 'win']).copy()
df_valid['win'] = df_valid['win'].astype(int)

# 3. 분석할 주요 지표 설정 (골드, 딜량, 시야, 팀내 딜 비중)
# 팀내 딜 비중(ch_teamDamagePercentage)은 역할 구조를 보는데 아주 좋은 지표입니다.
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
if 'ch_teamDamagePercentage' in df_valid.columns:
    metrics.append('ch_teamDamagePercentage')

# 4. 승패 및 포지션별 집계 (평균, 중앙값, 표준편차)
# 평균과 중앙값을 비교하면 분포 형태(쏠림)를 알 수 있고, 표준편차는 유저간 편차를 나타냅니다.
agg_funcs = ['mean', 'median', 'std']
grouped_stats = df_valid.groupby(['teamPosition', 'win'])[metrics].agg(agg_funcs)

# 멀티인덱스로 묶인 컬럼명을 하나로 평탄화 (예: goldEarned_mean)
grouped_stats.columns = [f"{col}_{func}" for col, func in grouped_stats.columns]
grouped_stats = grouped_stats.reset_index()

# 5. 보기 좋게 데이터 정제 (한글화 및 순서 정렬)
grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})

# 포지션 정렬 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])

# 컬럼명 직관적으로 변경하는 함수
def rename_columns(col_name):
    name = col_name
    name = name.replace('goldEarned', '골드').replace('totalDamageDealtToChampions', '딜량')
    name = name.replace('visionScore', '시야').replace('ch_teamDamagePercentage', '팀내_딜비중(%)')
    name = name.replace('_mean', '_평균').replace('_median', '_중앙값').replace('_std', '_표준편차')
    name = name.replace('teamPosition', '포지션').replace('win', '승패')
    return name

grouped_stats.rename(columns=lambda x: rename_columns(x), inplace=True)

# 소수점 반올림 (가독성을 위해)
numeric_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
grouped_stats[numeric_cols] = grouped_stats[numeric_cols].round(2)

# 6. 결과 확인
print("포지션 및 승패별 분포/구조/성과 지표:")
display(grouped_stats)

# 7. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv"
grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n분석용 데이터가 성공적으로 저장되었습니다:\n{output_path}")

포지션 및 승패별 분포/구조/성과 지표:


,포지션,승패,골드_평균,골드_중앙값,골드_표준편차,딜량_평균,딜량_중앙값,딜량_표준편차,시야_평균,시야_중앙값,시야_표준편차,팀내_딜비중(%)_평균,팀내_딜비중(%)_중앙값,팀내_딜비중(%)_표준편차
6,TOP,패배,10481.82,10415.0,4006.67,21112.24,19157.0,12294.36,17.19,16.0,9.92,0.23,0.22,0.08
7,TOP,승리,12229.37,12352.0,4034.34,25109.79,23521.0,13124.20,18.30,17.0,10.16,0.23,0.22,0.08
2,JUNGLE,패배,10939.27,10870.0,3960.13,17377.39,15521.5,11031.37,25.00,24.0,12.67,0.18,0.18,0.07
3,JUNGLE,승리,12752.84,12944.0,4140.90,20977.31,19403.0,12216.58,27.17,26.0,13.01,0.18,0.18,0.07
4,MIDDLE,패배,10602.92,10584.5,3939.09,21126.71,19381.0,12296.60,18.72,17.0,10.43,0.23,0.22,0.08
5,MIDDLE,승리,12164.72,12290.0,3941.59,25237.57,23843.0,13076.66,20.32,19.0,10.57,0.23,0.22,0.08
0,BOTTOM,패배,11471.68,11362.0,4450.99,21080.71,18643.0,13405.07,16.80,15.0,9.62,0.22,0.22,0.08
1,BOTTOM,승리,13488.46,13574.0,4550.86,26658.57,24559.0,14867.43,18.15,17.0,9.90,0.24,0.23,0.08
8,UTILITY,패배,7945.51,7859.0,2818.51,12447.67,10217.5,8866.17,60.34,59.0,31.07,0.14,0.12,0.07
9,UTILITY,승리,9035.45,9103.5,2851.07,13784.47,11561.5,9243.05,64.81,64.0,30.59,0.13,0.11,0.06



분석용 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv


In [ ]:
# 챔피언별 팀라인별 평균 골드,딜량,시야점수
import pandas as pd

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['championName', 'teamPosition']).copy()

# 3. 챔피언 및 포지션별 평균값 계산
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
grouped = df_valid.groupby(['championName', 'teamPosition'])[metrics].mean()

# 4. 포지션(teamPosition)을 열(Column)로 피벗(Pivot)
pivoted = grouped.unstack(level='teamPosition')

# 5. 원하는 열 순서와 이름 정의
positions = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
metric_mapping = {
    'goldEarned': '골드평균',
    'totalDamageDealtToChampions': '딜평균',
    'visionScore': '시야평균'
}

# 6. 지정된 순서대로 새로운 데이터프레임 구성
result_df = pd.DataFrame(index=pivoted.index)

for pos in positions:
    for orig_metric, new_metric in metric_mapping.items():
        # 데이터가 존재하는지 확인 후 삽입 (없으면 NaN)
        if (orig_metric, pos) in pivoted.columns:
            result_df[f'{pos}_{new_metric}'] = pivoted[(orig_metric, pos)]
        else:
            result_df[f'{pos}_{new_metric}'] = float('nan')

# 7. 데이터 정제 (결측치는 0으로 채우고 소수점 첫째 자리까지 반올림)
result_df = result_df.fillna(0).round(1)

# 8. 결과 확인 (상위 15개 챔피언 출력)
print("챔피언(행) x 포지션별 평균 지표(골드/딜/시야 순) 데이터:")
display(result_df.head(15))

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_metrics_ordered_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n정렬된 포지션별 평균 지표 데이터가 성공적으로 저장되었습니다:\n{output_path}")

챔피언(행) x 포지션별 평균 지표(골드/딜/시야 순) 데이터:


,TOP_골드평균,TOP_딜평균,TOP_시야평균,JUNGLE_골드평균,JUNGLE_딜평균,JUNGLE_시야평균,MIDDLE_골드평균,MIDDLE_딜평균,MIDDLE_시야평균,BOTTOM_골드평균,BOTTOM_딜평균,BOTTOM_시야평균,UTILITY_골드평균,UTILITY_딜평균,UTILITY_시야평균
championName,,,,,,,,,,,,,,,
Aatrox,10912.3,22881.5,17.3,11615.9,19560.0,25.3,10878.7,23347.0,17.4,8050.0,13482.0,8.5,8342.3,14749.1,44.3
Ahri,12563.6,27640.2,25.6,513.0,0.0,0.0,10968.8,21992.4,21.8,11091.0,17072.5,25.5,8786.4,16695.4,54.7
Akali,11549.7,25639.5,20.6,0.0,0.0,0.0,11266.1,24198.7,20.3,11548.2,21125.2,23.2,10275.5,21716.8,45.0
Akshan,13010.1,25547.7,21.2,0.0,0.0,0.0,12154.7,23662.9,19.5,9915.9,17909.5,13.1,9879.3,17615.0,38.2
Alistar,8724.2,16009.5,15.7,9475.3,9917.0,20.7,9765.2,16754.3,12.7,11895.3,17641.7,15.0,7766.6,8341.5,62.1
Ambessa,10976.5,23788.6,17.6,11759.7,18468.9,25.2,11083.7,23802.7,15.5,10064.2,19227.2,14.2,11450.5,28465.5,35.5
Amumu,7591.8,13293.2,10.2,10743.1,16557.9,22.5,0.0,0.0,0.0,0.0,0.0,0.0,8568.1,12333.7,57.5
Anivia,10667.4,21038.4,20.9,3689.0,2648.0,4.0,10901.6,20372.8,21.3,9641.7,11858.0,15.0,8694.6,14789.4,60.7
Annie,11184.8,26106.2,22.0,0.0,0.0,0.0,10842.4,24108.3,19.6,9272.8,16337.5,15.3,9330.9,19729.0,66.6



정렬된 포지션별 평균 지표 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_position_metrics_ordered_kr.csv


In [ ]:
#챔피언별 팀라인별 매치수 픽률/승률/평균(골드/딜/시야점수)
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거 및 데이터 타입 변환
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win']).copy()
df_valid['win'] = df_valid['win'].astype(int)

# 3. [기본 지표 계산] 매치 수, 픽률, 승률
match_count = pd.crosstab(index=df_valid['championName'], columns=df_valid['teamPosition'])
pick_rate = pd.crosstab(index=df_valid['championName'], columns=df_valid['teamPosition'], normalize='index') * 100
win_rate = pd.pivot_table(data=df_valid, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100

# 4. [심화 지표 계산] 골드, 딜량, 시야 점수 평균
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
grouped_avg = df_valid.groupby(['championName', 'teamPosition'])[metrics].mean()
pivoted_avg = grouped_avg.unstack(level='teamPosition')

# 5. 열 순서 지정 및 결측치 처리
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

# 6. 모든 지표를 하나의 데이터프레임으로 병합
result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    # 6-1. 기본 지표 추가
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)
    
    # 6-2. 심화 지표(골드, 딜, 시야) 추가
    # 데이터가 존재하는지 확인 후 삽입 (없으면 0.0)
    if ('goldEarned', pos) in pivoted_avg.columns:
        result_df[f'{pos}_골드평균'] = pivoted_avg[('goldEarned', pos)].fillna(0).round(1)
        result_df[f'{pos}_딜평균'] = pivoted_avg[('totalDamageDealtToChampions', pos)].fillna(0).round(1)
        result_df[f'{pos}_시야평균'] = pivoted_avg[('visionScore', pos)].fillna(0).round(1)
    else:
        result_df[f'{pos}_골드평균'] = 0.0
        result_df[f'{pos}_딜평균'] = 0.0
        result_df[f'{pos}_시야평균'] = 0.0

# 7. 결과 확인 (상위 15개 챔피언)
print("챔피언(행) x 포지션별 종합 지표 (매치수/픽률/승률/골드/딜/시야) 데이터:")
display(result_df.head(15))

# 8. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n모든 지표가 포함된 통합 마스터 데이터가 저장되었습니다:\n{output_path}")

챔피언(행) x 포지션별 종합 지표 (매치수/픽률/승률/골드/딜/시야) 데이터:


,TOP_매치수,TOP_픽률(%),TOP_승률(%),TOP_골드평균,TOP_딜평균,TOP_시야평균,JUNGLE_매치수,JUNGLE_픽률(%),JUNGLE_승률(%),JUNGLE_골드평균,...,BOTTOM_승률(%),BOTTOM_골드평균,BOTTOM_딜평균,BOTTOM_시야평균,UTILITY_매치수,UTILITY_픽률(%),UTILITY_승률(%),UTILITY_골드평균,UTILITY_딜평균,UTILITY_시야평균
championName,,,,,,,,,,,,,,,,,,,,,
Aatrox,1414,81.88,48.66,10912.3,22881.5,17.3,246,14.24,48.37,11615.9,...,100.00,8050.0,13482.0,8.5,11,0.64,27.27,8342.3,14749.1,44.3
Ahri,20,1.16,60.00,12563.6,27640.2,25.6,1,0.06,0.00,513.0,...,50.00,11091.0,17072.5,25.5,38,2.21,23.68,8786.4,16695.4,54.7
Akali,391,17.18,51.92,11549.7,25639.5,20.6,0,0.00,0.00,0.0,...,0.00,11548.2,21125.2,23.2,4,0.18,25.00,10275.5,21716.8,45.0
Akshan,12,4.65,50.00,13010.1,25547.7,21.2,0,0.00,0.00,0.0,...,12.50,9915.9,17909.5,13.1,6,2.33,50.00,9879.3,17615.0,38.2
Alistar,21,2.36,28.57,8724.2,16009.5,15.7,3,0.34,33.33,9475.3,...,0.00,11895.3,17641.7,15.0,852,95.73,50.82,7766.6,8341.5,62.1
Ambessa,956,79.53,47.70,10976.5,23788.6,17.6,177,14.73,51.98,11759.7,...,0.00,10064.2,19227.2,14.2,2,0.17,100.00,11450.5,28465.5,35.5
Amumu,4,0.68,25.00,7591.8,13293.2,10.2,462,79.11,55.19,10743.1,...,0.00,0.0,0.0,0.0,118,20.21,55.93,8568.1,12333.7,57.5
Anivia,34,7.66,44.12,10667.4,21038.4,20.9,1,0.23,0.00,3689.0,...,50.00,9641.7,11858.0,15.0,100,22.52,47.00,8694.6,14789.4,60.7
Annie,22,6.47,40.91,11184.8,26106.2,22.0,0,0.00,0.00,0.0,...,50.00,9272.8,16337.5,15.3,43,12.65,53.49,9330.9,19729.0,66.6



모든 지표가 포함된 통합 마스터 데이터가 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv
